# SC203 Colab runner

Use this notebook in the **Colab website**, not the Mac's local VS Code kernel. Run cells 2–5 in order. Cell 6 is an optional one-time asset check. Run cells 7–9 for a one-image smoke test. Run cell 10 for all 200 RealSR images only after the smoke test succeeds.

The repository branch is set in Cell 3. Push that branch from your Mac before pulling it here. Notebook outputs are cleared; do not save outputs when committing this notebook.


In [ ]:
# CELL 2 — Mount Google Drive. Run this before setup or asset checks.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 3 — Pull the exact code branch from GitHub. No token is used.
from pathlib import Path
import subprocess

BRANCH = "feat/baseline-runner"  # Change only when you intend to use a different pushed branch.
REPO_URL = "https://github.com/lemon4life/SC203.git"
REPO_DIR = Path("/content/SC203")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)

ASSETS_DIR = Path("/content/drive/MyDrive/lemmepass")
LOG_DIR = ASSETS_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
print(f"Repo ready: {REPO_DIR} @ {BRANCH} ({COMMIT})")


In [ ]:
# CELL 4 — Check Colab's Python and GPU. Stop here if CUDA is unavailable.
!nvidia-smi
import platform
import sys
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before continuing.")
print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# CELL 5 — Install/check DoSSR dependencies. Safe to rerun; preserves Colab's torch.
!{sys.executable} -u /content/SC203/scripts/colab_setup.py --requirements /content/SC203/external/DOSSR/requirements.txt --log-dir /content/drive/MyDrive/lemmepass/logs


In [ ]:
# CELL 6 — Optional asset check. Run once, or after changing assets; it loads the checkpoint on CPU.
# Already verified Phase 1 assets do not need to be checked every session.
# Uncomment the command below only when you want to verify them again.
# !{sys.executable} -u /content/SC203/scripts/verify_assets.py --assets-dir /content/drive/MyDrive/lemmepass --log-dir /content/drive/MyDrive/lemmepass/logs


In [ ]:
# CELL 7 — Prepare a true one-image smoke config. This copies one image to Colab disk.
# The helper changes input_dir and output_root in a temporary YAML file; it does not change the repo config.
!{sys.executable} -u /content/SC203/scripts/prepare_baseline_smoke.py --config /content/SC203/configs/baseline_realsr.yaml --smoke-dir /content/realsr_smoke --config-out /content/baseline_smoke.yaml --output-root /content/outputs/baseline-smoke --log-dir /content/drive/MyDrive/lemmepass/logs


In [ ]:
%%bash
# CELL 8 — Run DoSSR on the single smoke image. This still initializes the full model.
# Output is unbuffered and saved to Colab disk; the runner also writes a timestamped log to Drive.
cd /content/SC203
set -o pipefail
bash scripts/run_baseline.sh --config /content/baseline_smoke.yaml 2>&1 | tee /content/baseline_smoke_console.log
status=${PIPESTATUS[0]}
echo "SMOKE_RUNNER_EXIT_STATUS=$status"
exit 0


In [ ]:
# CELL 9 — Confirm smoke success before attempting the full 200-image baseline.
from pathlib import Path

smoke_output = Path("/content/outputs/baseline-smoke/seed_42")
smoke_images = sorted(smoke_output.glob("*.png")) if smoke_output.is_dir() else []
print(f"Smoke output folder: {smoke_output}")
print(f"PNG count: {len(smoke_images)}")
for image in smoke_images:
    print(image)
if len(smoke_images) != 1:
    print("Smoke test has not produced exactly one PNG; do not run the full baseline yet.")
else:
    print("SMOKE TEST PASSED — the full baseline cell is ready.")


In [ ]:
# CELL 10 — FULL BASELINE (200 images). Keep this disabled until Cell 9 reports SMOKE TEST PASSED.
# This can take a long time. The runner writes its timestamped log to Drive.
# Remove the leading # from the last line only when ready to run the full dataset.
# !set -o pipefail; bash /content/SC203/scripts/run_baseline.sh --config /content/SC203/configs/baseline_realsr.yaml 2>&1 | tee /content/baseline_full_console.log
